In [45]:
# adapted from MMCL (https://github.com/paulhager/MMCL-Tabular-Imaging)
import pandas as pd
import os
from os.path import join
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns
from torchvision.io import read_image
from matplotlib import pyplot as plt
import torchvision
import torch
import random
import numpy as np
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

pd.options.display.max_columns = 700

BASE = '/home/iml/marta.hasny/resized_DVM'
TABLES = '/home/iml/marta.hasny/dvm'
FEATURES = join(BASE, 'features')

front_view_only = False

from typing import List
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import k_means, SpectralClustering
import multiprocessing as mp

ANALYSIS = join(BASE, 'analysis')

def conf_matrix_from_matrices(mat_gt, mat_pred):
  overlap_and = (mat_pred & mat_gt)
  tp = overlap_and.sum()
  fp = mat_pred.sum()-overlap_and.sum()
  fn = mat_gt.sum()-overlap_and.sum()
  tn = mat_gt.shape[0]**2-(tp+fp+fn)
  return tp, fp, fn, tn

In [54]:
def check_or_save(obj, path, index=None, header=None):
  if isinstance(obj, pd.DataFrame):
    if index is None or header is None:
      raise ValueError('Index and header must be specified for saving a dataframe')
    if os.path.exists(path):
      if not header:
        saved_df = pd.read_csv(path,header=None)
      else:
        saved_df = pd.read_csv(path)
      naked_df = saved_df.reset_index(drop=True)
      naked_df.columns = range(naked_df.shape[1])
      naked_obj = obj.reset_index(drop=not index)
      naked_obj.columns = range(naked_obj.shape[1])
      if naked_df.round(6).equals(naked_obj.round(6)):
        return
      else:
        diff = (naked_df.round(6) == naked_obj.round(6))
        diff[naked_df.isnull()] = naked_df.isnull() & naked_obj.isnull()
        assert diff.all().all(), "Dataframe is not the same as saved dataframe"
    else:
      obj.to_csv(path, index=index, header=header)
  else:
    if os.path.exists(path):
      saved_obj = torch.load(path)
      if isinstance(obj, list):
        for i in range(len(obj)):
          check_array_equality(obj[i], saved_obj[i])
      else:
        check_array_equality(obj, saved_obj)
    else:
      print(f'Saving to {path}')
      torch.save(obj, path)


def check_array_equality(ob1, ob2):
  if torch.is_tensor(ob1) or isinstance(ob1, np.ndarray):
    assert (ob2 == ob1).all()
  else:
    assert ob2 == ob1

In [66]:
ad_data = pd.read_csv(join(TABLES, 'Ad_table.csv'))
ad_data.rename(columns={' Genmodel': 'Genmodel', ' Genmodel_ID': 'Genmodel_ID'}, inplace=True)

basic_data = pd.read_csv(join(TABLES, 'Basic_table.csv'))

image_data = pd.read_csv(join(TABLES, 'Image_table.csv'))
image_data.rename(columns={' Image_ID': 'Image_ID', ' Image_name': 'Image_name', ' Predicted_viewpoint':'Predicted_viewpoint', ' Quality_check':'Quality_check'}, inplace=True)

price_data = pd.read_csv(join(TABLES, 'Price_table.csv'))
price_data.rename(columns={' Genmodel': 'Genmodel', ' Genmodel_ID': 'Genmodel_ID', ' Year': 'Year', ' Entry_price': 'Entry_price'}, inplace=True)

sales_data = pd.read_csv(join(TABLES, 'Sales_table.csv'))
sales_data.rename(columns={'Genmodel ': 'Genmodel', 'Genmodel_ID ': 'Genmodel_ID'}, inplace=True)

trim_data = pd.read_csv(join(TABLES, 'Trim_table.csv'))

/tmp/ipykernel_2415727/622457649.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  ad_data = pd.read_csv(join(TABLES, 'Ad_table.csv'))


In [67]:
def parser_adv_id(x):
  split = x["Image_ID"].split('$$')
  return f"{split[0]}$${split[1]}"

image_data["Adv_ID"] = image_data.apply(lambda x: parser_adv_id(x), axis=1)
if front_view_only:
  image_data = image_data[(image_data["Quality_check"]=="P")&(image_data["Predicted_viewpoint"]==0)]
image_data.drop_duplicates(subset=['Adv_ID'], inplace=True)
image_data

,Genmodel_ID,Image_ID,Image_name,Predicted_viewpoint,Quality_check,Adv_ID
0,2_1,2_1$$1$$1,Abarth$$124 Spider$$2017$$Blue$$2_1$$1$$image_...,45,NaN,2_1$$1
1,2_1,2_1$$10$$11,Abarth$$124 Spider$$2017$$Blue$$2_1$$10$$image...,45,NaN,2_1$$10
8,2_1,2_1$$4$$0,Abarth$$124 Spider$$2017$$Blue$$2_1$$4$$image_...,0,P,2_1$$4
14,2_1,2_1$$8$$3,Abarth$$124 Spider$$2017$$Blue$$2_1$$8$$image_...,0,NaN,2_1$$8
18,2_1,2_1$$13$$8,Abarth$$124 Spider$$2017$$Grey$$2_1$$13$$image...,0,P,2_1$$13
...,...,...,...,...,...,...
1451766,96_18,96_18$$919$$3,Volvo$$XC90$$2019$$White$$96_18$$919$$image_3.jpg,225,NaN,96_18$$919
1451771,97_1,97_1$$1$$1,Westfield$$Sport$$2006$$Yellow$$97_1$$1$$image...,45,NaN,97_1$$1
1451772,99_1,99_1$$2$$14,Zenos$$E10$$2016$$Green$$99_1$$2$$image_14.jpg,180,NaN,99_1$$2
1451775,99_1,99_1$$3$$1,Zenos$$E10$$2016$$Grey$$99_1$$3$$image_1.jpg,0,P,99_1$$3


In [68]:
print(len(ad_data))
feature_df = ad_data.merge(price_data[['Genmodel_ID', 'Entry_price', 'Year']], left_on=['Genmodel_ID','Reg_year'], right_on=['Genmodel_ID','Year'])
print(len(feature_df))
feature_df

268255
224724


,Maker,Genmodel,Genmodel_ID,Adv_ID,Adv_year,Adv_month,Color,Reg_year,Bodytype,Runned_Miles,Engin_size,Gearbox,Fuel_type,Price,Seat_num,Door_num,Entry_price,Year
0,Bentley,Arnage,10_1,10_1$$1,2018,4,Silver,2000.0,Saloon,60000,6.8L,Automatic,Petrol,21500,5.0,4.0,145000,2000
1,Bentley,Arnage,10_1,10_1$$2,2018,6,Grey,2002.0,Saloon,44000,6.8L,Automatic,Petrol,28750,5.0,4.0,149000,2002
2,Bentley,Arnage,10_1,10_1$$3,2017,11,Blue,2002.0,Saloon,55000,6.8L,Automatic,Petrol,29999,5.0,4.0,149000,2002
3,Bentley,Arnage,10_1,10_1$$4,2018,4,Green,2003.0,Saloon,14000,6.8L,Automatic,Petrol,34948,5.0,4.0,151500,2003
4,Bentley,Arnage,10_1,10_1$$5,2017,11,Grey,2003.0,Saloon,61652,6.8L,Automatic,Petrol,26555,5.0,4.0,151500,2003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224719,Volvo,V50,96_9,96_9$$522,2018,7,Grey,2008.0,Estate,140000,2.0L,Manual,Diesel,3185,5.0,5.0,15780,2008
224720,Volvo,V50,96_9,96_9$$523,2018,8,Blue,2007.0,Estate,158000,2.4L,Automatic,Diesel,2990,5.0,5.0,15780,2007
224721,Volvo,V50,96_9,96_9$$524,2018,5,Silver,2009.0,Estate,94000,2.4L,Automatic,Diesel,4250,5.0,5.0,15770,2009
224722,Volvo,V50,96_9,96_9$$525,2018,5,Silver,2004.0,Estate,111000,2.4L,Automatic,Petrol,2895,5.0,5.0,17165,2004


In [69]:
data_df = feature_df.merge(image_data[['Adv_ID', 'Image_name', 'Predicted_viewpoint']], left_on=['Adv_ID'], right_on=['Adv_ID'])
assert data_df["Adv_ID"].is_unique

In [70]:
def extract_engine_size(x):
  return float(x['Engin_size'][:-1])

data_df.dropna(inplace=True)
data_df['Engine_size'] = data_df.apply(lambda x: extract_engine_size(x), axis=1)
data_df.drop(columns=['Engin_size'], inplace=True)
data_df

,Maker,Genmodel,Genmodel_ID,Adv_ID,Adv_year,Adv_month,Color,Reg_year,Bodytype,Runned_Miles,Gearbox,Fuel_type,Price,Seat_num,Door_num,Entry_price,Year,Image_name,Predicted_viewpoint,Engine_size
0,Bentley,Arnage,10_1,10_1$$1,2018,4,Silver,2000.0,Saloon,60000,Automatic,Petrol,21500,5.0,4.0,145000,2000,Bentley$$Arnage$$2000$$Silver$$10_1$$1$$image_...,45,6.8
1,Bentley,Arnage,10_1,10_1$$2,2018,6,Grey,2002.0,Saloon,44000,Automatic,Petrol,28750,5.0,4.0,149000,2002,Bentley$$Arnage$$2002$$Grey$$10_1$$2$$image_0.jpg,0,6.8
2,Bentley,Arnage,10_1,10_1$$3,2017,11,Blue,2002.0,Saloon,55000,Automatic,Petrol,29999,5.0,4.0,149000,2002,Bentley$$Arnage$$2002$$Blue$$10_1$$3$$image_0.jpg,90,6.8
3,Bentley,Arnage,10_1,10_1$$4,2018,4,Green,2003.0,Saloon,14000,Automatic,Petrol,34948,5.0,4.0,151500,2003,Bentley$$Arnage$$2003$$Green$$10_1$$4$$image_0...,315,6.8
4,Bentley,Arnage,10_1,10_1$$5,2017,11,Grey,2003.0,Saloon,61652,Automatic,Petrol,26555,5.0,4.0,151500,2003,Bentley$$Arnage$$2003$$Grey$$10_1$$5$$image_10...,0,6.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209030,Volvo,V50,96_9,96_9$$521,2018,4,Silver,2007.0,Estate,88452,Manual,Diesel,3450,5.0,5.0,15780,2007,Volvo$$V50$$2007$$Silver$$96_9$$521$$image_0.jpg,225,2.0
209031,Volvo,V50,96_9,96_9$$523,2018,8,Blue,2007.0,Estate,158000,Automatic,Diesel,2990,5.0,5.0,15780,2007,Volvo$$V50$$2007$$Blue$$96_9$$523$$image_0.jpg,315,2.4
209032,Volvo,V50,96_9,96_9$$524,2018,5,Silver,2009.0,Estate,94000,Automatic,Diesel,4250,5.0,5.0,15770,2009,Volvo$$V50$$2009$$Silver$$96_9$$524$$image_0.jpg,135,2.4
209033,Volvo,V50,96_9,96_9$$525,2018,5,Silver,2004.0,Estate,111000,Automatic,Petrol,2895,5.0,5.0,17165,2004,Volvo$$V50$$2004$$Silver$$96_9$$525$$image_0.jpg,135,2.4


In [71]:
id_df = data_df.loc[:,'Adv_ID']
image_name_df = data_df.loc[:,'Image_name']
viewpoint_df = data_df.loc[:,'Predicted_viewpoint']

continuous_df = data_df.loc[:,(
  'Adv_year',
  'Adv_month',
  'Reg_year',
  'Runned_Miles',
  'Price',
  'Seat_num',
  'Door_num',
  'Entry_price', 
  'Engine_size'
  )]

categorical_ids = [
  'Color',
  'Bodytype',
  'Gearbox',
  'Fuel_type',
  'Genmodel_ID'
  ]

categorical_df = data_df.loc[:,categorical_ids]

continuous_df['Runned_Miles'] = pd.to_numeric(continuous_df['Runned_Miles'], errors='coerce')
continuous_df['Price'] = pd.to_numeric(continuous_df['Price'], errors='coerce')

# normalize
continuous_df=(continuous_df-continuous_df.mean())/continuous_df.std()

categorical_df['Color'] = categorical_df['Color'].astype('category')
categorical_df['Bodytype'] = categorical_df['Bodytype'].astype('category')
categorical_df['Gearbox'] = categorical_df['Gearbox'].astype('category')
categorical_df['Fuel_type'] = categorical_df['Fuel_type'].astype('category')
categorical_df['Genmodel_ID'] = categorical_df['Genmodel_ID'].astype('category')

cat_columns = categorical_df.select_dtypes(['category']).columns

categorical_df[cat_columns] = categorical_df[cat_columns].apply(lambda x: x.cat.codes)

data_df = pd.concat([id_df, continuous_df, categorical_df, image_name_df, viewpoint_df], axis=1)
data_df.dropna(inplace=True)

In [72]:
minimum_population = 100
values = (data_df.value_counts(subset=['Genmodel_ID'])>=minimum_population).values
codes = (data_df.value_counts(subset=['Genmodel_ID'])>=minimum_population).index
populated_codes = []
for i, v in enumerate(values):
  if v:
    populated_codes.append(int(codes[i][0]))

In [73]:
len(populated_codes)

286

In [74]:
data_df = data_df[data_df['Genmodel_ID'].isin(populated_codes)]
map = {}
for i,l in enumerate(data_df['Genmodel_ID'].unique()):
  map[l] = i
data_df['Genmodel_ID'] = data_df['Genmodel_ID'].map(map)
data_df

,Adv_ID,Adv_year,Adv_month,Reg_year,Runned_Miles,Price,Seat_num,Door_num,Entry_price,Engine_size,Color,Bodytype,Gearbox,Fuel_type,Genmodel_ID,Image_name,Predicted_viewpoint
28,10_3$$1,0.012811,-1.311853,0.876887,-0.906021,6.323766,0.135216,0.61833,6.285820,5.331307,8,10,0,8,0,Bentley$$Bentayga$$2016$$Grey$$10_3$$1$$image_...,225
29,10_3$$2,0.012811,-0.830929,1.339829,-1.192392,8.212201,0.135216,0.61833,6.285820,5.331307,4,10,0,8,0,Bentley$$Bentayga$$2018$$Brown$$10_3$$2$$image...,90
30,10_3$$3,0.012811,-0.830929,0.876887,-0.848026,6.594044,0.135216,0.61833,6.285820,5.331307,18,10,0,8,0,Bentley$$Bentayga$$2016$$Silver$$10_3$$3$$imag...,225
31,10_3$$4,0.012811,-1.792777,1.339829,-1.131942,8.198984,0.135216,0.61833,6.285820,2.739476,8,10,0,1,0,Bentley$$Bentayga$$2018$$Grey$$10_3$$4$$image_...,90
33,10_3$$6,0.012811,-0.350005,1.339829,-1.190405,8.428261,0.135216,0.61833,6.285820,2.739476,1,10,0,8,0,Bentley$$Bentayga$$2018$$Black$$10_3$$6$$image...,315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209030,96_9$$521,0.012811,-0.830929,-1.206348,0.866347,-0.502760,0.135216,0.61833,-0.279830,0.147646,18,4,1,1,285,Volvo$$V50$$2007$$Silver$$96_9$$521$$image_0.jpg,225
209031,96_9$$523,0.012811,1.092766,-1.206348,2.492723,-0.527576,0.135216,0.61833,-0.279830,0.666012,2,4,0,1,285,Volvo$$V50$$2007$$Blue$$96_9$$523$$image_0.jpg,315
209032,96_9$$524,0.012811,-0.350005,-0.743407,0.996087,-0.459602,0.135216,0.61833,-0.280377,0.666012,18,4,0,1,285,Volvo$$V50$$2009$$Silver$$96_9$$524$$image_0.jpg,135
209033,96_9$$525,0.012811,-0.350005,-1.900760,1.393631,-0.532701,0.135216,0.61833,-0.204064,0.666012,18,4,0,8,285,Volvo$$V50$$2004$$Silver$$96_9$$525$$image_0.jpg,135


In [75]:
bad_indices = []
for indx, row in data_df.iterrows():
    im_name = row['Image_name']
    split = im_name.split('$$')
    path = join(BASE, 'resized_DVM', split[0], split[1], split[2], split[3], im_name)
    if not os.path.exists(path):
        bad_indices.append(path)

In [84]:
_ids = list(data_df['Adv_ID'])
addendum = '_all_views'

non_feature_columns = ['Adv_ID', 'Adv_year', 'Adv_month', 'Reg_year', 'Runned_Miles', 'Color', 'Image_name', 'Predicted_viewpoint', 'Genmodel_ID']
if front_view_only:
  train_set_ids, test_ids = train_test_split(_ids, test_size=0.1, random_state=2022)
  train_ids, val_ids = train_test_split(train_set_ids, test_size=0.2, random_state=2022)
  
  bad_indices_train = torch.load(join(FEATURES, f'bad_indices_train{addendum}.pt'))
  bad_indices_val = torch.load(join(FEATURES, f'bad_indices_val{addendum}.pt'))

  print(f'Val length before {len(val_ids)}')
  for _id in bad_indices_val:
      val_ids.remove(_id)
  print(f'Val length after {len(val_ids)}')

  print(f'Train length before {len(train_ids)}')
  for _id in bad_indices_train:
      train_ids.remove(_id)
  print(f'Train length after {len(train_ids)}')
else:
  addendum = '_all_views'
  train_set_ids, test_ids = train_test_split(_ids, test_size=0.5, random_state=2022, stratify=data_df['Genmodel_ID'])
  train_ids, val_ids = train_test_split(train_set_ids, test_size=0.2, random_state=2022, stratify=data_df[data_df['Adv_ID'].isin(train_set_ids)]['Genmodel_ID'])

check_or_save(train_ids, join(FEATURES, f'train_ids{addendum}.pt'))
check_or_save(val_ids, join(FEATURES, f'val_ids{addendum}.pt'))
check_or_save(test_ids, join(FEATURES, f'test_ids{addendum}.pt'))

train_df = data_df.set_index('Adv_ID').loc[train_ids]
val_df = data_df.set_index('Adv_ID').loc[val_ids]
test_df = data_df.set_index('Adv_ID').loc[test_ids]

train_labels_all = list(train_df['Genmodel_ID'])
val_labels_all = list(val_df['Genmodel_ID'])
test_labels_all = list(test_df['Genmodel_ID'])

check_or_save(train_labels_all, join(FEATURES,f'labels_model_all_train{addendum}.pt'))
check_or_save(val_labels_all, join(FEATURES,f'labels_model_all_val{addendum}.pt'))
check_or_save(test_labels_all, join(FEATURES,f'labels_model_all_test{addendum}.pt'))

check_or_save(train_df.loc[:,~train_df.columns.isin(non_feature_columns)],join(FEATURES,f'dvm_features_train_noOH{addendum}.csv'), index=False, header=False)
check_or_save(val_df.loc[:,~val_df.columns.isin(non_feature_columns)],join(FEATURES,f'dvm_features_val_noOH{addendum}.csv'), index=False, header=False)
check_or_save(test_df.loc[:,~test_df.columns.isin(non_feature_columns)],join(FEATURES,f'dvm_features_test_noOH{addendum}.csv'), index=False, header=False)

check_or_save(train_df, join(FEATURES,f'dvm_full_features_train_noOH{addendum}.csv'), index=True, header=True)
check_or_save(val_df, join(FEATURES,f'dvm_full_features_val_noOH{addendum}.csv'), index=True, header=True)
check_or_save(test_df, join(FEATURES,f'dvm_full_features_test_noOH{addendum}.csv'), index=True, header=True)

lengths = [1 for i in range(len(continuous_df.columns))]

if 'Genmodel_ID' in categorical_ids:
  categorical_ids.remove('Genmodel_ID')
if 'Color' in categorical_ids:
  categorical_ids.remove('Color')
max = list(data_df[categorical_ids].max(axis=0))
max = [i+1 for i in max]
lengths = lengths + max
check_or_save(lengths, join(FEATURES, f'tabular_lengths{addendum}.pt'))

Saving to /home/iml/marta.hasny/resized_DVM/features/train_ids_all_views.pt
Saving to /home/iml/marta.hasny/resized_DVM/features/val_ids_all_views.pt
Saving to /home/iml/marta.hasny/resized_DVM/features/test_ids_all_views.pt
Saving to /home/iml/marta.hasny/resized_DVM/features/labels_model_all_train_all_views.pt
Saving to /home/iml/marta.hasny/resized_DVM/features/labels_model_all_val_all_views.pt
Saving to /home/iml/marta.hasny/resized_DVM/features/labels_model_all_test_all_views.pt
Saving to /home/iml/marta.hasny/resized_DVM/features/tabular_lengths_all_views.pt


In [87]:
def get_paths(df):
  paths = []
  for indx, row in df.iterrows():
      im_name = row['Image_name']
      split = im_name.split('$$')
      path = join(BASE, 'resized_DVM', split[0], split[1], split[2], split[3], im_name)
      paths.append(path)
  return paths

# For big dataset need to save only paths to load live
addendum = '_all_views'
train_df = pd.read_csv(join(FEATURES,f'dvm_full_features_train_noOH{addendum}.csv'))
val_df = pd.read_csv(join(FEATURES,f'dvm_full_features_val_noOH{addendum}.csv'))
test_df = pd.read_csv(join(FEATURES,f'dvm_full_features_test_noOH{addendum}.csv'))

for df, name in zip([train_df, val_df, test_df], ['train', 'val', 'test']):
  paths = get_paths(df)
  check_or_save(paths, join(FEATURES, f'{name}_paths{addendum}.pt'))

Saving to /home/iml/marta.hasny/resized_DVM/features/train_paths_all_views.pt
Saving to /home/iml/marta.hasny/resized_DVM/features/val_paths_all_views.pt
Saving to /home/iml/marta.hasny/resized_DVM/features/test_paths_all_views.pt


# New Physical Features

## Adding missing values to physical table

In [88]:
# Fill using other values
physical_df_orig = pd.read_csv(join(TABLES,'Ad_table (extra).csv'))
physical_df_orig.rename(columns={' Genmodel_ID':'Genmodel_ID', ' Genmodel':'Genmodel'}, inplace=True)

# Manual touches

# Peugeot RCZ
physical_df_orig.loc[physical_df_orig['Genmodel_ID'] == '69_36','Wheelbase']=2612
# Ford Grand C-Max
physical_df_orig.loc[physical_df_orig['Genmodel_ID'] == '29_20','Wheelbase']=2788 

def fill_from_other_entry(row):
    for attr in ['Wheelbase', 'Length', 'Width', 'Height']:
        if pd.isna(row[attr]) or row[attr]==0:
            other_rows = physical_df_orig.loc[physical_df_orig['Genmodel_ID']==row['Genmodel_ID']]
            other_rows.dropna(subset=[attr], inplace=True)
            other_rows.drop_duplicates(subset=[attr], inplace=True)
            other_rows = other_rows[other_rows[attr]>0]
            if len(other_rows)>0:
                row[attr] = other_rows[attr].values[0]
    return row

physical_df_orig = physical_df_orig.apply(fill_from_other_entry, axis=1)

physical_df_orig.to_csv(join(FEATURES,'Ad_table_physical_filled.csv'), index=False)

## Add physical attributes to features

In [89]:
# Add jitter to physical dimensions so they aren't just labels
def add_jitter(x, jitter=50):
    return x + random.randint(-jitter, jitter)

random.seed(2022)
physical_df = pd.read_csv(join(FEATURES,'Ad_table_physical_filled.csv'))
for attr in ['Wheelbase', 'Length', 'Width', 'Height']:
    physical_df[attr] = physical_df[attr].apply(add_jitter)
physical_df.to_csv(join(FEATURES,'Ad_table_physical_filled_jittered_50.csv'), index=False)

In [90]:
# Ford ranger (29_30) has wrong height. Missing 1 in front... 805.0 instead of 1805.0
# Mercedes Benz (59_29) wrong wheelbase, 5246.0 instead of 3106
# Kia Rio (43_9) wrong wheelbase, 4065.0 instead of 2580
# FIXED


physical_df = pd.read_csv(join(FEATURES,'Ad_table_physical_filled.csv'))[['Adv_ID', 'Wheelbase','Height','Width','Length']]
for v in ['_all_views']:
    for split in ['train', 'val', 'test']:
        features_df = pd.read_csv(join(FEATURES,f'dvm_full_features_{split}_noOH{v}.csv'))
        merged_df = features_df.merge(physical_df, on='Adv_ID')
        physical_only_df = merged_df[['Wheelbase','Height','Width','Length','Bodytype']]

        for attr in ['Wheelbase','Height','Width','Length']:
            assert merged_df[attr].isna().sum()==0
            assert (merged_df[attr]==0).sum()==0

        # normalize physical attributes
        for attr in ['Wheelbase','Height','Width','Length']:
            merged_df[attr] = (merged_df[attr]-merged_df[attr].mean())/merged_df[attr].std()
            physical_only_df[attr] = (physical_only_df[attr]-physical_only_df[attr].mean())/physical_only_df[attr].std()

        # Drop unwanted cols
        non_feature_columns = ['Adv_ID', 'Image_name', 'Genmodel_ID']
        if v == '_all_views':
            non_feature_columns.append('Predicted_viewpoint')
        merged_df = merged_df.drop(non_feature_columns, axis=1)

        merged_df_cols = merged_df.columns.tolist()
        rearranged_cols = merged_df_cols[-4:]+merged_df_cols[:-4]
        merged_df = merged_df[rearranged_cols]
        check_or_save(merged_df, join(FEATURES,f'dvm_features_{split}_noOH{v}_physical.csv'), index=False, header=True)
        check_or_save(physical_only_df, join(FEATURES,f'dvm_features_{split}_noOH{v}_physical_only.csv'), index=False, header=False)
    lengths = torch.load(join(FEATURES,f'tabular_lengths{v}.pt'))
    new_lengths = [1,1,1,1]
    lengths = new_lengths + lengths
    check_or_save(lengths, join(FEATURES,f'tabular_lengths{v}_physical.pt'))
    lengths = [1,1,1,1,13]
    check_or_save(lengths, join(FEATURES,f'tabular_lengths{v}_physical_only.pt'))

/tmp/ipykernel_2415727/2366573916.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  physical_only_df[attr] = (physical_only_df[attr]-physical_only_df[attr].mean())/physical_only_df[attr].std()
/tmp/ipykernel_2415727/2366573916.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  physical_only_df[attr] = (physical_only_df[attr]-physical_only_df[attr].mean())/physical_only_df[attr].std()
/tmp/ipykernel_2415727/2366573916.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

Saving to /home/iml/marta.hasny/resized_DVM/features/tabular_lengths_all_views_physical.pt
Saving to /home/iml/marta.hasny/resized_DVM/features/tabular_lengths_all_views_physical_only.pt


# Add Labels to Features

In [91]:
for v in ['_all_views']:
    for split in ['train', 'val']:
        labels = torch.load(join(FEATURES,f'labels_model_all_{split}{v}.pt'))
        features = pd.read_csv(join(FEATURES,f'dvm_features_{split}_noOH{v}_physical.csv'))
        features['label'] = labels
        features.to_csv(join(FEATURES,f'dvm_features_{split}_noOH{v}_physical_labeled.csv'), index=False)
        # check_or_save(features, join(FEATURES,f'dvm_features_{split}_noOH{v}_physical_labeled.csv'), index=False, header=True)
    lengths = torch.load(join(FEATURES,f'tabular_lengths{v}_physical.pt'))
    lengths.append(np.max(labels)+1)
    check_or_save(lengths, join(FEATURES,f'tabular_lengths{v}_physical_labeled.pt'))

Saving to /home/iml/marta.hasny/resized_DVM/features/tabular_lengths_all_views_physical_labeled.pt


# Price Label

In [92]:
df = pd.read_csv(os.path.join(FEATURES, 'Ad_table_physical_filled.csv'))
train_paths = torch.load(os.path.join(FEATURES, 'train_paths_all_views.pt'))
val_paths = torch.load(os.path.join(FEATURES, 'val_paths_all_views.pt'))
test_paths = torch.load(os.path.join(FEATURES, 'test_paths_all_views.pt'))

In [93]:
def get_prices(paths):
    price_map = df.set_index('Adv_ID')['Price'].to_dict()
    
    prices = []
    for p in tqdm(paths):
        filename = p.split('/')[-1]
        parts = filename.split('$$')
        adv_id = parts[-3] + '$$' + parts[-2]
        prices.append(price_map.get(adv_id))
        
    return prices

In [94]:
train_prices = get_prices(train_paths)
val_prices = get_prices(val_paths)
test_prices = get_prices(test_paths)

100%|██████████| 88207/88207 [00:00<00:00, 913485.17it/s]


In [95]:
train_tensor = torch.tensor(train_prices, dtype=torch.float)
val_tensor = torch.tensor(val_prices, dtype=torch.float)
test_tensor = torch.tensor(test_prices, dtype=torch.float)
torch.save(train_tensor, os.path.join(FEATURES, "prices_train.pt"))
torch.save(val_tensor, os.path.join(FEATURES, "prices_val.pt"))
torch.save(test_tensor, os.path.join(FEATURES, "prices_test.pt"))

In [99]:
train_prices = np.array(train_prices)
val_prices = np.array(val_prices)
test_prices = np.array(test_prices)
mu = train_prices.mean()
sigma = train_prices.std()
train_prices_norm = (train_prices - mu) / sigma
val_prices_norm = (val_prices - mu) / sigma
test_prices_norm = (test_prices - mu) / sigma

In [101]:
train_tensor_norm = torch.tensor(train_prices_norm, dtype=torch.float)
val_tensor_norm = torch.tensor(val_prices_norm, dtype=torch.float)
test_tensor_norm = torch.tensor(test_prices_norm, dtype=torch.float)
torch.save(train_tensor_norm, os.path.join(FEATURES, "prices_train_norm.pt"))
torch.save(val_tensor_norm, os.path.join(FEATURES, "prices_val_norm.pt"))
torch.save(test_tensor_norm, os.path.join(FEATURES, "prices_test_norm.pt"))